In [ ]:
# 無法run成功，因為缺少必要的套件和模型權重檔案。
# 請確保已安裝 Keras 和相關的依賴項，並且已下載 ResNet50 模型的權重檔案。
# 此外，請確認圖片輸入的格式和大小符合模型的要求。

from keras.applications.resnet50 import ResNet50
from keras.applications.resnet50 import preprocess_input
from keras.applications.resnet50 import decode_predictions
from PIL import Image
import numpy as np
import gradio as gr

# 建立 RasNet50 模型
model = ResNet50(weights="imagenet", include_top=True) 

def resize_image(img, new_w, new_h):
    # 轉換成 PIL 圖形
    img = Image.fromarray(img)    
    # 取得圖片的原始尺寸
    w, h = img.size
    # 計算長寬比例
    w_scale = new_w / w
    h_scale = new_h / h 
    scale = min(w_scale, h_scale)
    # 調整圖片尺寸
    resized = img.resize((int(w*scale), int(h*scale)), Image.NEAREST)    
    # 剪裁成正確的尺寸
    resized = resized.crop((0, 0, new_w, new_h))
    return resized

def predict(input):
    input_resized = resize_image(input, 224, 224)
    # 轉換圖片成為NumPy陣列
    img = np.array(input_resized)
    # 轉換成4D張量(1, 244, 244, 3)
    img = img.reshape((1, 224, 224, 3))
    # 資料預處理
    img = preprocess_input(img)    
    # 使用模型進行預測
    y_pred = model.predict(img, verbose=0)
    # 解碼預測結果
    label = decode_predictions(y_pred)
    max_len = len(label[0])
    max_len = 10 if max_len > 10 else max_len
    # 產生前10個結果的 Python 字典
    top_10_predictions = {
        label[0][i][1]: float(label[0][i][2])
        for i in range(max_len)
    }

    return top_10_predictions

inputs = gr.Image()
outputs = gr.Label(num_top_classes=3)    
app = gr.Interface(fn=predict, 
                   inputs=inputs, 
                   outputs=outputs)
app.launch()

Running on local URL:  http://127.0.0.1:7864

To create a public link, set `share=True` in `launch()`.
